# Video -> Product Detection -> Ecommerce Recommendation Pipeline

This notebook takes a **video** (from the current folder, a URL, or a public S3 link),
detects **distinct products** in it (ignoring humans), finds the **exact product** on
trusted ecommerce sites at a **configurable match threshold** (the `0.90` requirement),
uploads the detected crops to **S3**, and writes a **timestamped metadata file** so you
can overlay recommendations on the video timeline.

**Everything is configured in `config.yaml`.** Edit that file, then run this notebook top to bottom.

### Honest expectations (please read)
- No detector or matcher is *100% accurate*. This uses state-of-the-art open models and makes every threshold tunable so you can push precision as high as possible.
- Getting the **exact** product (the specific watch, not just "a watch") requires a visual-search backend that indexes ecommerce catalogs. The default is **SerpApi Google Lens `exact_matches`** — the reliable, no-ban path. A fully-offline CLIP catalog backend is included too.
- Direct Google Images scraping gets blocked by Google; that's policy, not a bug. SerpApi + on-disk caching + retry/backoff is how we avoid rate limits and crashes.

## 1. Setup
Install dependencies (run once). Install the CUDA build of torch first to use your GPU.

In [ ]:
# Run once. For GPU, install the matching CUDA torch build BEFORE this, e.g.:
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
%pip install -q -r requirements.txt

In [ ]:
import importlib
import pipeline_utils as pu
importlib.reload(pu)   # so edits to the helper module take effect without restarting

pu.load_env('.env')                 # load API keys / AWS creds
cfg = pu.load_config('config.yaml') # single source of truth for all settings

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Matching         : Google Lens (SerpApi), lens_type =', pu.get(cfg, 'matching.serpapi.lens_type'))
print('Match threshold  :', pu.get(cfg, 'matching.min_match_score'))
print('Detection backend:', pu.get(cfg, 'detection.backend'), '|', pu.get(cfg, 'detection.model_weights'))

## 2. Ingest the video
Set `input.source_type` in `config.yaml` to `local`, `url`, or `s3`. All three work.

In [ ]:
video_path = pu.ingest_video(cfg)
video_id = pu.slugify(video_path.stem)
print('Local video path:', video_path)
print('Video id (used for S3 prefix + metadata):', video_id)

## 3. Sample frames
Pulls one frame every `frames.sample_every_seconds`, optionally within a time window,
skipping near-identical consecutive frames to save compute.

In [ ]:
frames = pu.sample_frames(video_path, cfg)
print(f'Sampled {len(frames)} frames for analysis.')
if frames:
    print('First frame at %.2fs, last at %.2fs' % (frames[0].timestamp, frames[-1].timestamp))

## 4. Detect products (open-vocabulary, humans ignored)
Uses YOLOE / YOLO-World. Labels in `detection.ignore_labels` (person, hand, face, ...) are dropped.
Each surviving detection is cropped and saved locally.

In [ ]:
from tqdm.auto import tqdm

detector = pu.Detector(cfg)
crops_dir = pu.get(cfg, 's3.local_crops_dir', 'output/crops')

all_detections = []
for fr in tqdm(frames, desc='Detecting'):
    dets = detector.detect_frame(fr)
    for d in dets:
        detector.save_crop(fr, d, crops_dir)
        all_detections.append(d)

print(f'{len(all_detections)} product detections (humans already filtered out).')
from collections import Counter
print('By label:', dict(Counter(d.label for d in all_detections)))

## 5. Deduplicate into DISTINCT products
CLIP-embeds every crop, then greedily clusters same-label crops whose cosine similarity
exceeds `dedup.same_product_similarity`. One representative crop per distinct product.

In [ ]:
embedder = pu.Embedder(cfg, section='dedup')
crop_paths = [d.crop_path for d in all_detections]
embeddings = embedder.embed_image_paths(crop_paths)

products = pu.dedup_products(all_detections, embeddings, cfg)
print(f'{len(products)} DISTINCT products after dedup.')
for p in products:
    print(f'  {p.product_id}: {p.label}  seen {p.first_seen:.1f}s->{p.last_seen:.1f}s  '
          f'({len(p.occurrences)} occurrences)')

### (optional) Preview the distinct product crops

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

n = len(products)
if n:
    cols = min(4, n); rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 3, rows * 3))
    for i, p in enumerate(products):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(Image.open(p.representative_crop))
        plt.title(f'{p.product_id}: {p.label}', fontsize=9)
        plt.axis('off')
    plt.tight_layout(); plt.show()

## 6. Upload crops to S3
Done **before** matching because Google Lens needs a publicly reachable image URL.
The returned URL (public or presigned) is stored on each product and reused for matching.

In [ ]:
uploader = pu.S3Uploader(cfg, video_id=video_id)
if uploader.enabled:
    uploader.preflight()   # clear error if bucket/creds/region are wrong
    for p in tqdm(products, desc='Uploading to S3'):
        # The uploader forces the real file extension, so the base name is enough.
        suffix = f'{p.product_id}_{pu.slugify(p.label)}'
        p.s3_url = uploader.upload(p.representative_crop, key_suffix=suffix)
    print('Uploaded', len(products), 'crops. Example URL:')
    print(' ', products[0].s3_url[:120] if products else '(none)')
else:
    print('S3 disabled in config (s3.enabled: false); skipping upload.')

## 7. Match each distinct product to EXACT ecommerce listings
Uses **Google Lens** (via SerpApi). Only recommendations at/above
`matching.min_match_score` (your **0.90**) and from `matching.trusted_domains` are kept.
Results are cached on disk, so re-running never re-queries (and never re-bills).

In [ ]:
# Google Lens backend; the embedder arg is unused here but kept for compatibility.
matcher = pu.Matcher(cfg, embedder=embedder)

for p in tqdm(products, desc='Matching'):
    image_url = p.s3_url or None   # required by SerpApi Google Lens
    try:
        p.recommendations = matcher.match(p, image_url=image_url)
    except Exception as e:
        print(f'[warn] matching failed for {p.product_id} ({p.label}): {e}')
        p.recommendations = []

for p in products:
    print(f'\n{p.product_id} [{p.label}] -> {len(p.recommendations)} matches >= '
          f'{pu.get(cfg, "matching.min_match_score")}')
    for r in p.recommendations:
        print(f'   {r.score:.2f}  {r.title[:60]!r}  {r.price}  {r.source}')
        print(f'         {r.url}')

## 8. Write the timestamped metadata file
Produces `output/detections.json` (full data) and, if enabled, `output/detections.vtt`
(a WebVTT track you can drop straight into an HTML5 `<video>` to overlay recommendations).

In [ ]:
import cv2
cap = cv2.VideoCapture(str(video_path))
video_info = {
    'id': video_id,
    'path': str(video_path),
    'fps': cap.get(cv2.CAP_PROP_FPS),
    'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0),
    'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0),
    'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0),
}
video_info['duration_seconds'] = (video_info['frame_count'] / video_info['fps']) if video_info['fps'] else None
cap.release()

payload = pu.write_metadata(products, video_info, cfg)
print('Wrote', pu.get(cfg, 'metadata.output_path'))
if pu.get(cfg, 'metadata.emit_webvtt'):
    print('Wrote', pu.get(cfg, 'metadata.webvtt_path'))
print(f"\nSummary: {payload['product_count']} distinct products with recommendations.")

## 9. (optional) Peek at the metadata JSON

In [ ]:
import json
print(json.dumps(payload, indent=2)[:2500])

---
### Tuning cheat-sheet (all in `config.yaml`)
- **Stricter "exact" matches** → raise `matching.min_match_score` (e.g. 0.95) and keep `serpapi.lens_type: exact_matches`.
- **More recall / more products** → lower `detection.confidence_threshold` and `frames.sample_every_seconds`.
- **Fewer duplicate products** → lower `dedup.same_product_similarity`; **more distinct variants** → raise it.
- **Restrict to specific stores** → edit `matching.trusted_domains`.
- **Looser / more matches** → set `matching.serpapi.lens_type: visual_matches` or lower `min_match_score`.
- **Rate-limit safety** → tune `network.min_interval_seconds`, `network.max_retries`, `network.backoff_base_seconds`.